### All MySQL commands using Python for automating MySQL

In [ ]:
import mysql.connector

DB_CONFIG = dict(host="localhost", user="root", password="garvit@123", database="aegis_neo")

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

def run(sql, description=""):
    conn = get_conn()
    cur = conn.cursor()
    try:
        cur.execute(sql)
        conn.commit()
        print(f"✅ {description or 'OK'} | rows affected: {cur.rowcount}")
    except Exception as e:
        print(f"❌ {description}: {e}")
    finally:
        cur.close(); conn.close()

def fetch(sql):
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    cur.close(); conn.close()
    return cols, rows

In [ ]:
run("""
CREATE TABLE IF NOT EXISTS dim_neo (
    neo_id VARCHAR(20) PRIMARY KEY,
    full_name VARCHAR(150),
    is_hazardous BOOLEAN,
    diameter_km_avg FLOAT,
    absolute_magnitude_h FLOAT,
    eccentricity FLOAT,
    semi_major_axis_au FLOAT,
    inclination_deg FLOAT,
    orbital_period_days FLOAT,
    data_arc_days INT,
    n_observations INT,
    impact_probability FLOAT,
    palermo_scale_max FLOAT,
    torino_scale INT,
    last_obs_date VARCHAR(20)
)
""", "create dim_neo")

run("""
CREATE TABLE IF NOT EXISTS fact_close_approach (
    approach_id INT AUTO_INCREMENT PRIMARY KEY,
    neo_id VARCHAR(20),
    close_approach_date DATE,
    relative_velocity_kmh FLOAT,
    miss_distance_km FLOAT,
    miss_distance_ld FLOAT,
    orbiting_body VARCHAR(50),
    FOREIGN KEY (neo_id) REFERENCES dim_neo(neo_id)
)
""", "create fact_close_approach")

In [ ]:
run("""
INSERT INTO dim_neo (neo_id, full_name, diameter_km_avg, absolute_magnitude_h,
    eccentricity, semi_major_axis_au, inclination_deg, orbital_period_days,
    data_arc_days, n_observations, impact_probability, palermo_scale_max,
    torino_scale, last_obs_date, is_hazardous)
SELECT
    o.neo_id, TRIM(o.full_name),
    AVG((f.est_diameter_min_km + f.est_diameter_max_km) / 2),
    NULL, o.eccentricity, o.semi_major_axis_au, o.inclination_deg,
    o.orbital_period_days, o.data_arc_days, o.n_observations,
    s.impact_probability, s.palermo_scale_max, s.torino_scale,
    s.last_obs_date, MAX(f.is_hazardous)
FROM raw_orbital_elements o
LEFT JOIN raw_sentry_risk s ON o.neo_id = s.neo_id
LEFT JOIN raw_neo_feed f ON o.neo_id = f.neo_id
GROUP BY o.neo_id, o.full_name, o.eccentricity, o.semi_major_axis_au,
         o.inclination_deg, o.orbital_period_days, o.data_arc_days,
         o.n_observations, s.impact_probability, s.palermo_scale_max,
         s.torino_scale, s.last_obs_date
""", "populate dim_neo")

In [ ]:
run("""
INSERT INTO fact_close_approach (neo_id, close_approach_date, relative_velocity_kmh,
    miss_distance_km, miss_distance_ld, orbiting_body)
SELECT f.neo_id, f.close_approach_date, f.relative_velocity_kmh, f.miss_distance_km,
    f.miss_distance_km / 384400, f.orbiting_body
FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.neo_id = d.neo_id
""", "populate fact_close_approach")

In [ ]:
cols, rows = fetch("SELECT COUNT(*) FROM raw_neo_feed")
print("raw_neo_feed total:", rows[0][0])
cols, rows = fetch("SELECT COUNT(DISTINCT neo_id) FROM raw_neo_feed")
print("raw_neo_feed unique neo_id:", rows[0][0])
cols, rows = fetch("SELECT COUNT(*) FROM fact_close_approach")
print("fact_close_approach inserted:", rows[0][0])


In [ ]:
import pandas as pd

In [ ]:
cols, rows = fetch("SELECT neo_id, name FROM raw_neo_feed LIMIT 5")
print(pd.DataFrame(rows, columns=cols))

cols, rows = fetch("SELECT neo_id, full_name FROM dim_neo LIMIT 5")
print(pd.DataFrame(rows, columns=cols))

In [ ]:
run("ALTER TABLE raw_neo_feed ADD COLUMN designation_num VARCHAR(20)", "add designation_num to raw_neo_feed")
run("ALTER TABLE dim_neo ADD COLUMN designation_num VARCHAR(20)", "add designation_num to dim_neo")

run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(name), ' ', 1))
""", "populate designation_num in raw_neo_feed")

run("""
UPDATE dim_neo
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(full_name), ' ', 1))
""", "populate designation_num in dim_neo")

In [ ]:
cols, rows = fetch("""
SELECT COUNT(*) FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.designation_num = d.designation_num
""")
print("matched rows:", rows[0][0])

In [ ]:
import requests
import mysql.connector

API_KEY = "hBnrFPk07qCcnQy1unyYO2VDgD1nAKH9gi8liBHu"
DB_CONFIG = dict(host="localhost", user="root", password="garvit@123", database="aegis_neo")

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

def fetch_neo_feed(start_date, end_date):
    url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date={start_date}&end_date={end_date}&api_key={API_KEY}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()["near_earth_objects"]

def insert_feed_data(data):
    conn = get_conn()
    cur = conn.cursor()
    for date, objects in data.items():
        for obj in objects:
            approach = obj["close_approach_data"][0]
            cur.execute("""
                INSERT IGNORE INTO raw_neo_feed
                (neo_id, name, absolute_magnitude_h, est_diameter_min_km, est_diameter_max_km,
                 is_hazardous, close_approach_date, relative_velocity_kmh, miss_distance_km, orbiting_body)
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
            """, (
                obj["id"], obj["name"], obj["absolute_magnitude_h"],
                obj["estimated_diameter"]["kilometers"]["estimated_diameter_min"],
                obj["estimated_diameter"]["kilometers"]["estimated_diameter_max"],
                obj["is_potentially_hazardous_asteroid"], date,
                float(approach["relative_velocity"]["kilometers_per_hour"]),
                float(approach["miss_distance"]["kilometers"]),
                approach["orbiting_body"]
            ))
    conn.commit()
    cur.close(); conn.close()

In [ ]:
import time
from datetime import datetime, timedelta

today = datetime.today()
for i in range(0, 180, 7):  # ~6 months, week by week
    start = (today - timedelta(days=i+7)).strftime("%Y-%m-%d")
    end = (today - timedelta(days=i)).strftime("%Y-%m-%d")
    try:
        insert_feed_data(fetch_neo_feed(start, end))
        print(f"✅ {start} to {end}")
    except Exception as e:
        print(f"❌ {start} to {end}: {e}")
    time.sleep(1)

In [ ]:
import requests

def fetch_orbital_by_designation(designation):
    url = "https://ssd-api.jpl.nasa.gov/sbdb.api"
    params = {"sstr": designation, "full-prec": "true"}
    r = requests.get(url, params=params, timeout=20)
    if r.status_code != 200:
        return None
    return r.json()

# get distinct designations from your (now much bigger) feed table
cols, rows = fetch("SELECT DISTINCT designation_num FROM raw_neo_feed")
designations = [r[0] for r in rows]
print(f"{len(designations)} unique objects to fetch")

In [ ]:
run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(name), ' ', 1))
WHERE designation_num IS NULL
""", "backfill designation_num for new rows")

In [ ]:
cols, rows = fetch("SELECT COUNT(*) FROM raw_neo_feed")
print("total feed rows:", rows[0][0])

cols, rows = fetch("SELECT DISTINCT designation_num FROM raw_neo_feed")
designations = [r[0] for r in rows]
print(f"{len(designations)} unique objects to fetch")

In [ ]:
import requests
import time

def fetch_orbital_by_designation(designation):
    url = "https://ssd-api.jpl.nasa.gov/sbdb.api"
    params = {"sstr": designation, "full-prec": "true"}
    r = requests.get(url, params=params, timeout=20)
    if r.status_code != 200:
        return None
    return r.json()

results = {}
for i, des in enumerate(designations):
    try:
        data = fetch_orbital_by_designation(des)
        if data and "orbit" in data:
            results[des] = data
            print(f"✅ {i+1}/{len(designations)} {des}")
        else:
            print(f"⚠️ {i+1}/{len(designations)} {des} — no orbit data")
    except Exception as e:
        print(f"❌ {des}: {e}")
    time.sleep(0.5)  # respect rate limit

In [ ]:
import re

def clean_designation(name):
    name = name.strip()
    if name.startswith("("):
        # provisional designation, e.g. "(2020 AB1)" -> "2020 AB1"
        return name.strip("()")
    else:
        # numbered asteroid, e.g. "240320 (2003 HS42)" -> "240320"
        return name.split(" ")[0]

# rebuild designations list correctly from raw_neo_feed
cols, rows = fetch("SELECT DISTINCT name FROM raw_neo_feed")
designations = [clean_designation(r[0]) for r in rows]
print(f"{len(designations)} unique objects to fetch")
print(designations[:10])

In [ ]:
run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(TRAILING ')' FROM TRIM(LEADING '(' FROM TRIM(name)))
WHERE name LIKE '(%'
""", "fix provisional designation_num")

run("""
UPDATE raw_neo_feed
SET designation_num = TRIM(SUBSTRING_INDEX(TRIM(name), ' ', 1))
WHERE name NOT LIKE '(%'
""", "fix numbered designation_num")

In [ ]:
cols, rows = fetch("SELECT DISTINCT name FROM raw_neo_feed")
designations = [clean_designation(r[0]) for r in rows]
print(f"{len(designations)} unique objects to fetch")

: 

In [ ]:
results = {}
for i, des in enumerate(designations):
    try:
        data = fetch_orbital_by_designation(des)
        if data and "orbit" in data:
            results[des] = data
            print(f"✅ {i+1}/{len(designations)} {des}")
        else:
            print(f"⚠️ {i+1}/{len(designations)} {des} — no orbit data")
    except Exception as e:
        print(f"❌ {des}: {e}")
    time.sleep(0.5)

In [ ]:
def insert_dim_neo_from_sbdb(results):
    conn = get_conn()
    cur = conn.cursor()
    for des, data in results.items():
        obj = data.get("object", {})
        orbit = data.get("orbit", {})
        elems = {e["name"]: e["value"] for e in orbit.get("elements", [])}
        spkid = obj.get("spkid")
        full_name = obj.get("fullname")

        cur.execute("""
            INSERT INTO dim_neo
            (neo_id, full_name, designation_num, eccentricity, semi_major_axis_au,
             inclination_deg, orbital_period_days, data_arc_days, n_observations)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
            ON DUPLICATE KEY UPDATE full_name=VALUES(full_name)
        """, (
            spkid, full_name, des,
            float(elems.get("e", 0) or 0),
            float(elems.get("a", 0) or 0),
            float(elems.get("i", 0) or 0),
            float(elems.get("per", 0) or 0),
            None, None
        ))
    conn.commit()
    cur.close(); conn.close()

insert_dim_neo_from_sbdb(results)

In [ ]:
insert_dim_neo_from_sbdb(results)

In [ ]:
cols, rows = fetch("SELECT COUNT(*) FROM dim_neo")
print("dim_neo rows:", rows[0][0])

cols, rows = fetch("""
SELECT COUNT(*) FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.designation_num = d.designation_num
""")
print("matched rows for fact table:", rows[0][0])

In [ ]:
run("ALTER TABLE dim_neo ADD COLUMN priority_score FLOAT", "add priority_score column")

run("""
UPDATE dim_neo
SET priority_score = 
    (COALESCE(impact_probability,0) * 100000) +
    (CASE WHEN is_hazardous = 1 THEN 20 ELSE 0 END) +
    (CASE WHEN data_arc_days < 365 THEN 15 ELSE 0 END) +
    (CASE WHEN n_observations < 50 THEN 15 ELSE 0 END) +
    (CASE WHEN torino_scale > 0 THEN 30 ELSE 0 END)
""", "compute priority_score")

In [ ]:
cols, rows = fetch("SELECT COUNT(*) FROM dim_neo")
print("dim_neo rows:", rows[0][0])

cols, rows = fetch("SELECT COUNT(*) FROM fact_close_approach")
print("fact_close_approach rows:", rows[0][0])

cols, rows = fetch("SELECT full_name, priority_score FROM dim_neo ORDER BY priority_score DESC LIMIT 10")
import pandas as pd
print(pd.DataFrame(rows, columns=cols))

In [ ]:
run("""
INSERT INTO fact_close_approach (neo_id, close_approach_date, relative_velocity_kmh,
    miss_distance_km, miss_distance_ld, orbiting_body)
SELECT d.neo_id, f.close_approach_date, f.relative_velocity_kmh, f.miss_distance_km,
    f.miss_distance_km / 384400, f.orbiting_body
FROM raw_neo_feed f
INNER JOIN dim_neo d ON f.designation_num = d.designation_num
""", "populate fact_close_approach")

In [ ]:
cols, rows = fetch("SELECT COUNT(*) FROM fact_close_approach")
print("fact_close_approach rows:", rows[0][0])

In [ ]:
run("""
UPDATE dim_neo d
INNER JOIN raw_neo_feed f ON d.designation_num = f.designation_num
SET d.is_hazardous = f.is_hazardous
WHERE d.is_hazardous IS NULL
""", "backfill is_hazardous")

In [ ]:
run("""
UPDATE dim_neo
SET priority_score = 
    (COALESCE(impact_probability,0) * 100000) +
    (CASE WHEN is_hazardous = 1 THEN 20 ELSE 0 END) +
    (CASE WHEN data_arc_days < 365 THEN 15 ELSE 0 END) +
    (CASE WHEN n_observations < 50 THEN 15 ELSE 0 END) +
    (CASE WHEN torino_scale > 0 THEN 30 ELSE 0 END)
""", "recompute priority_score")

In [ ]:
cols, rows = fetch("SELECT full_name, priority_score FROM dim_neo ORDER BY priority_score DESC LIMIT 10")
print(pd.DataFrame(rows, columns=cols))